In [1]:
import sys
sys.path.insert(0, '/home/aiscuser/verl')

In [2]:
import asyncio
import time
from typing import AsyncIterator, Dict, List, Optional, Tuple, Union
import uuid
from sglang.srt.managers.io_struct import GenerateReqInput, AbortReq
from sglang.srt.entrypoints.verl_engine import VerlEngine as VerlEngineBase
from sglang.srt.entrypoints.verl_engine import _preprocess_tensor_for_update_weights
from sglang.srt.server import Engine
from sglang.srt.utils import MultiprocessingSerializer, broadcast_pyobj
from sglang.srt.model_executor.model_runner import LocalSerializedTensor

import os

import torch
import torch.distributed as dist
from torch.distributed.tensor import DeviceMesh, DTensor

from verl.third_party.sglang.entrypoint import CustomEngine


/opt/conda/envs/ptca/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 04-12 18:24:18 __init__.py:190] Automatically detected platform cuda.


In [3]:
engine = CustomEngine(
    model_path='Qwen/Qwen2.5-3B',
    tp_size=4,
    node_rank=0,
    nnodes=1,
)

INFO 04-12 18:25:56 __init__.py:190] Automatically detected platform cuda.
INFO 04-12 18:25:56 __init__.py:190] Automatically detected platform cuda.
INFO 04-12 18:25:56 __init__.py:190] Automatically detected platform cuda.
INFO 04-12 18:25:56 __init__.py:190] Automatically detected platform cuda.
INFO 04-12 18:25:57 __init__.py:190] Automatically detected platform cuda.


[W412 18:26:00.543797968 Utils.hpp:135] Warning: Environment variable NCCL_ASYNC_ERROR_HANDLING is deprecated; use TORCH_NCCL_ASYNC_ERROR_HANDLING instead (function operator())
[W412 18:26:00.878578564 Utils.hpp:135] Warning: Environment variable NCCL_ASYNC_ERROR_HANDLING is deprecated; use TORCH_NCCL_ASYNC_ERROR_HANDLING instead (function operator())
[W412 18:26:00.339962148 Utils.hpp:135] Warning: Environment variable NCCL_ASYNC_ERROR_HANDLING is deprecated; use TORCH_NCCL_ASYNC_ERROR_HANDLING instead (function operator())
[W412 18:26:00.343632680 Utils.hpp:135] Warning: Environment variable NCCL_ASYNC_ERROR_HANDLING is deprecated; use TORCH_NCCL_ASYNC_ERROR_HANDLING instead (function operator())


node-0:2891126:2891126 [0] NCCL INFO Bootstrap : Using eth0:10.1.74.7<0>
node-0:2891126:2891126 [0] NCCL INFO NET/Plugin: Failed to find ncclNetPlugin_v8 symbol.
node-0:2891126:2891126 [0] NCCL INFO NET/Plugin: Loaded net plugin NCCL RDMA Plugin v6 (v6)
node-0:2891126:2891126 [0] NCCL INFO NET/Plugin: Failed to find ncclCollNetPlugin_v8 symbol.
node-0:2891126:2891126 [0] NCCL INFO NET/Plugin: Failed to find ncclCollNetPlugin symbol (>= v5). ncclCollNetPlugin symbols v4 and lower are not supported.
node-0:2891126:2891126 [0] NCCL INFO cudaDriverVersion 12040
NCCL version 2.21.5+cuda12.4
node-0:2891128:2891128 [2] NCCL INFO cudaDriverVersion 12040
node-0:2891128:2891128 [2] NCCL INFO Bootstrap : Using eth0:10.1.74.7<0>
node-0:2891127:2891127 [1] NCCL INFO cudaDriverVersion 12040
node-0:2891127:2891127 [1] NCCL INFO Bootstrap : Using eth0:10.1.74.7<0>
node-0:2891129:2891129 [3] NCCL INFO cudaDriverVersion 12040
node-0:2891129:2891129 [3] NCCL INFO Bootstrap : Using eth0:10.1.74.7<0>
node-

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:00<00:00,  8.36it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:00<00:00,  5.83it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:00<00:00,  6.11it/s]

100%|██████████| 23/23 [00:08<00:00,  2.58it/s]


In [28]:
async def get_first_n_results(self, tasks, num_returns):
    outputs = {}
    completed_oids = []
    completed_rids = []
    all_tasks = [task for task_dict in tasks.values() for task in task_dict.values()]
    for oid in tasks.keys():
        outputs[oid] = {}
    for task in asyncio.as_completed(all_tasks):
        result = await task
        rid = result['meta_info']['id']
        oid = rid.split('_nid')[0]
        outputs[oid][rid] = result
        completed_rids.append(rid)
        tasks[oid].pop(rid)
        if len(tasks[oid].keys()) == 0:
            completed_oids.append(oid)
            tasks.pop(oid)
        if len(completed_oids) >= num_returns:
            break

    cached_states = []

    for oid, task_dict in tasks.items():
        for rid in task_dict.keys():
            if rid in self.tokenizer_manager.rid_to_state:
                cached_states.append(self.tokenizer_manager.rid_to_state[rid])
            self.tokenizer_manager.abort_request(rid)

    # Wait for idle
    while True:
        # print(f'waiting for idle')
        internal_state = await self.tokenizer_manager.get_internal_state()
        if internal_state['is_idle']:
            # print(f'idle')
            break
    
    # Scavenge incomplete results
    incomplete_tasks = [task for task_dict in tasks.values() for task in task_dict.values()]

    await asyncio.sleep(1)
    for task in incomplete_tasks:
        if task.done():
            result = await task
            incomplete_tasks.remove(task)
            rid = result['meta_info']['id']
            oid = rid.split('_nid')[0]
            outputs[oid][rid] = result
            finish_reason = result['meta_info'].get('finish_reason', {type: ''})
            # to_delete_rids.remove(rid)
            if finish_reason['type'] != 'abort':
                completed_rids.append(rid)
                tasks[oid].pop(rid)
                if len(tasks[oid].keys()) == 0:
                    completed_oids.append(oid)
                    tasks.pop(oid)
        else:
            task.cancel()

    partial_outputs = self.gather_partial_outputs()

    self.tokenizer_manager.clear_queue()
    for output in partial_outputs:
        rid = output['meta_info']['id']
        oid = rid.split('_nid')[0]
        outputs[oid][rid] = output

    return outputs, cached_states, partial_outputs

engine.get_first_n_results = get_first_n_results.__get__(engine, CustomEngine)

In [5]:
from transformers import AutoTokenizer, AutoProcessor

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B')
processor = AutoProcessor.from_pretrained('Qwen/Qwen2.5-3B')

from verl.utils.dataset import RLHFDataset
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/aime-2024.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    image_key='images',
)

dataset len: 960


In [22]:
def get_batch(batch_size):
    batch = []
    rid = []
    for i in range(batch_size):
        batch.append(dataset[i]['raw_prompt_ids'])
        rid.append(f'{i}_nid{uuid.uuid4().hex[:8]}')
    return batch, rid

sampling_params = {
    'n': 1,
    'temperature': 1.0,
    'top_p': 1.0,
    'top_k': -1,
    'max_new_tokens': 1024,
}


In [29]:
batch, rid = get_batch(128)
outputs, cached_states, partial_outputs = engine.custom_generate(
    input_ids=batch,
    sampling_params=sampling_params,
    return_logprob=True,
    rid=rid,
    num_returns=32,
)

Clear queue
Clear queue
Clear queue
Clear queue


In [32]:
list(outputs.values())[0]

{'45_nidca0ed7c2': {'text': 'Answer:347',
  'meta_info': {'id': '45_nidca0ed7c2',
   'finish_reason': {'type': 'stop', 'matched': 151643},
   'prompt_tokens': 119,
   'input_token_logprobs': [(None, 198, None)],
   'output_token_logprobs': [(-1.218550205230713, 16141, None),
    (-0.04230528697371483, 25, None),
    (-8.621325492858887, 18, None),
    (-2.408630132675171, 19, None),
    (-3.941506862640381, 22, None),
    (-0.7494242787361145, 151643, None)],
   'completion_tokens': 6,
   'cached_tokens': 118,
   'e2e_latency': 0.41875410079956055}}}

In [27]:
for oid, out_dict in outputs.items():
    for rid, output in out_dict.items():
        finish_reason = output['meta_info']['finish_reason']
        finished = finish_reason['type'] != 'abort'
        if not finished:
            print(output)

{'output_ids': [1249, 11625, 419, 3491, 11, 582, 1184, 311, 990, 279, 17508, 315, 27187, 10187, 8957, 13, 5692, 748, 279, 3019, 14319, 29208, 29985, 1447, 16, 13, 3070, 35338, 21419, 25, 1019, 256, 481, 6771, 17767, 422, 1124, 8, 387, 279, 738, 315, 10826, 879, 1828, 264, 22205, 10058, 624, 256, 481, 6771, 17767, 356, 1124, 8, 387, 279, 738, 315, 10826, 879, 1828, 264, 738, 315, 19120, 18890, 624, 256, 481, 6771, 17767, 328, 1124, 8, 387, 279, 738, 315, 10826, 879, 1828, 264, 13551, 978, 1021, 624, 256, 481, 6771, 17767, 425, 1124, 8, 387, 279, 738, 315, 10826, 879, 1828, 264, 8968, 315, 31556, 22662, 382, 17, 13, 3070, 22043, 24979, 25, 1019, 256, 481, 17767, 760, 35, 91, 284, 220, 16, 24, 20, 1124, 340, 256, 481, 17767, 760, 34, 91, 284, 220, 18, 21, 22, 1124, 340, 256, 481, 17767, 760, 50, 91, 284, 220, 20, 21, 17, 1124, 340, 256, 481, 17767, 760, 33, 91, 284, 220, 24, 15, 15, 1124, 692, 18, 13, 3070, 72927, 315, 9043, 12525, 25, 1019, 256, 481, 2619, 525, 220, 19, 18, 22, 10826, 87